Weil das Skript IDW_Umrechnung_Verkehrsdistanz nur jeweils die Verkehrsmesstation in nächster Nähe zu den Luftmessensoren berücksichtigt, greift dieses Skript auf alle Verkehrsmessstationen zu um das Maximale aus dem Datensatz rauszuholen. Je nach Distanz (wieder IDW) und je nach Fahrzeugtyp (PKW oder Truck) werden zwei Spalten mit kumultativen Werten zu dem Einfluss auf die Luftqualtität ausgegeben.

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
base_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Sensor_Community)"
verkehr_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Verkehrsmesswerte)\Verkehrsdaten_mit_Koordinaten_merged_cars_n_trucks.csv"
sensor_raw_file = os.path.join(
    base_path, "sensor_community_complete_may_august.csv"
)

# Deine aktuelle Master-ML-Datei (bitte den exakten Namen anpassen, falls abweichend)
ml_master_file = os.path.join(base_path, "FINAL_ML_Ready_Dataset_Wuerzburg_GIS_PBLH.csv")
output_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\kumulativ_traffic_ML_Ready_Dataset_Wuerzburg.csv"


def calculate_distance_matrix_m(lat1, lon1, lat2_series, lon2_series):
    """Berechnet die Distanz in Metern von einem Punkt zu einer ganzen Serie von Koordinaten."""
    R = 6371000.0
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2_series)
    dphi = np.radians(lat2_series - lat1)
    dlambda = np.radians(lon2_series - lon1)

    a = (
        np.sin(dphi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    )
    return R * 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


# =========================================================
# 2. Verkehrsdaten für 13:00 Uhr laden & numerisch absichern
# =========================================================
print("Lade und bereinige Verkehrsdaten (13:00 Uhr)...")
df_traffic = pd.read_csv(verkehr_file, sep=";", decimal=",")

# Zeitstempel formatieren und strikt auf 13 Uhr filtern
df_traffic["Time"] = pd.to_datetime(
    df_traffic["Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce"
)
df_traffic_13 = df_traffic[df_traffic["Time"].dt.hour == 13].copy()
df_traffic_13["date"] = df_traffic_13["Time"].dt.strftime("%Y-%m-%d")

# Komma-Zahlen kugelsicher in Floats umwandeln
for col in ["Lat", "Lon", "processed_car_vol", "processed_truck_vol"]:
    df_traffic_13[col] = (
        df_traffic_13[col].astype(str).str.replace(",", ".", regex=False)
    )
    df_traffic_13[col] = pd.to_numeric(df_traffic_13[col], errors="coerce")

df_traffic_13 = df_traffic_13.dropna(
    subset=["Lat", "Lon", "processed_car_vol"]
)
print(f"-> {len(df_traffic_13)} Verkehrs-Beobachtungen um 13:00 Uhr geladen.")

# =========================================================
# 3. Sensor-Koordinaten extrahieren
# =========================================================
print("Extrahiere Sensor-Koordinaten...")
df_sc = pd.read_csv(sensor_raw_file, sep=";", decimal=",")

# Koordinaten bereinigen
for col in ["lat", "lon"]:
    df_sc[col] = df_sc[col].astype(str).str.replace(",", ".", regex=False)
    df_sc[col] = pd.to_numeric(df_sc[col], errors="coerce")

sensor_coords = (
    df_sc[["sensor_id", "lat", "lon"]].drop_duplicates(subset=["sensor_id"]).dropna()
)
sensor_coords["station_id"] = (
    sensor_coords["sensor_id"].astype(float).astype(int).astype(str)
)
sensor_coords = sensor_coords[["station_id", "lat", "lon"]]

# WICHTIG: Die offizielle Station "Würzburg/Kopfklinik" als Anker manuell hinzufügen,
# falls sie nicht in den Sensor.Community-Rohdaten steht!
kopfklinik_entry = pd.DataFrame(
    [
        {
            "station_id": "Würzburg/Kopfklinik",
            "lat": 49.8047,
            "lon": 9.95642,
        }
    ]
)
all_sensors = pd.concat(
    [sensor_coords, kopfklinik_entry], ignore_index=True
).drop_duplicates(subset=["station_id"])
print(
    f"-> {len(all_sensors)} Sensor-Standorte (inkl. Kopfklinik) verriegelt."
)

# =========================================================
# 4. Kumulativen Proximity-Index pro Sensor & Tag berechnen
# =========================================================
print(
    "Berechne kumulative Verkehrs-Proxys über alle 30+ Verkehrsstationen..."
)
cumulative_results = []

# Wir iterieren über jeden Tag im Verkehrs-Datensatz
unique_dates = df_traffic_13["date"].unique()

for current_date in unique_dates:
    # Verkehrsnetz an genau diesem Tag um 13:00 Uhr isolieren
    daily_traffic = df_traffic_13[df_traffic_13["date"] == current_date]

    for _, sensor in all_sensors.iterrows():
        s_id = sensor["station_id"]
        s_lat = sensor["lat"]
        s_lon = sensor["lon"]

        # Distanz von diesem Sensor zu ALLEN aktiven Verkehrsstellen an diesem Tag berechnen
        distances = calculate_distance_matrix_m(
            s_lat, s_lon, daily_traffic["Lat"], daily_traffic["Lon"]
        )

        # Inverse Distance Weighting (IDW): Volumen / (Distanz + 1)
        idw_all_series = daily_traffic["processed_car_vol"] / (
            distances + 1.0
        )
        idw_truck_series = daily_traffic["processed_truck_vol"].fillna(0.0) / (
            distances + 1.0
        )

        # Die magische Summe über die gesamte Stadt
        kumulativ_all = idw_all_series.sum()
        kumulativ_truck = idw_truck_series.sum()

        cumulative_results.append(
            {
                "date": current_date,
                "station_id": s_id,
                "prox_kumulativ_car": round(kumulativ_all, 4),
                "prox_kumulativ_truck": round(kumulativ_truck, 4),
            }
        )

df_cum_proxy = pd.DataFrame(cumulative_results)
print(f"-> {len(df_cum_proxy)} Tag-Sensor-Kombinationen berechnet.")

# =========================================================
# 5. Zusammenführen mit dem Master-ML-Datensatz
# =========================================================
print("Integriere neue Features in den ML-Datensatz...")
df_ml = pd.read_csv(ml_master_file, sep=";", decimal=",")

# Falls beim Einlesen Kommas waren, versuche Standard-Komma-Trennung
if len(df_ml.columns) <= 1:
    df_ml = pd.read_csv(ml_master_file, sep=",")

df_ml["station_id"] = df_ml["station_id"].astype(str)
df_cum_proxy["station_id"] = df_cum_proxy["station_id"].astype(str)

# Datum einheitlich auf YYYY-MM-DD formatieren
df_ml["date_clean"] = pd.to_datetime(
    df_ml["date"], format="mixed", dayfirst=True, errors="coerce"
).dt.strftime("%Y-%m-%d")

# Alte, fehlerhafte oder einzelne Verkehrsspalten rigoros löschen (verhindert _x / _y Spalten!)
cols_to_remove = [
    c
    for c in df_ml.columns
    if any(
        x in c
        for x in [
            "verkehrsaufkommen",
            "distanz_verkehr",
            "prox_",
            "traffic",
            "_x",
            "_y",
        ]
    )
]
df_ml_clean = df_ml.drop(columns=cols_to_remove, errors="ignore")

# Finaler Merge über Datum und Sensor-ID
df_final = pd.merge(
    df_ml_clean,
    df_cum_proxy,
    left_on=["date_clean", "station_id"],
    right_on=["date", "station_id"],
    how="left",
)

# Hilfsspalte aufräumen
#df_final = df_final.drop(columns=["date_clean"], errors="ignore")

# Speichern
df_final.to_csv(output_file, index=False, sep=";", decimal=",")
print("-" * 60)
print(
    f"ERFOLG! Der finale Datensatz mit kumulativen Verkehrs-Features liegt hier:\n{output_file}"
)
print("-" * 60)
display(
    df_final[
        [
            "date_clean",
            "station_id",
            "PM25_Target",
            "prox_kumulativ_car",
            "prox_kumulativ_truck",
        ]
    ].head(10)
)

Lade und bereinige Verkehrsdaten (13:00 Uhr)...
-> 11808 Verkehrs-Beobachtungen um 13:00 Uhr geladen.
Extrahiere Sensor-Koordinaten...
-> 9 Sensor-Standorte (inkl. Kopfklinik) verriegelt.
Berechne kumulative Verkehrs-Proxys über alle 30+ Verkehrsstationen...
-> 1107 Tag-Sensor-Kombinationen berechnet.
Integriere neue Features in den ML-Datensatz...
------------------------------------------------------------
ERFOLG! Der finale Datensatz mit kumulativen Verkehrs-Features liegt hier:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\kumulativ_traffic_ML_Ready_Dataset_Wuerzburg.csv
------------------------------------------------------------


,date_clean,station_id,PM25_Target,prox_kumulativ_car,prox_kumulativ_truck
0,2024-05-01,10645,3.853571,11.4868,0.2273
1,2024-05-01,19623,4.301020,14.1929,0.2905
2,2024-05-01,25049,5.718261,7.0257,0.1374
3,2024-05-02,10645,3.251190,16.7927,0.9354
4,2024-05-02,19623,4.165918,20.7154,1.2670
5,2024-05-02,25049,5.332041,10.5255,0.6279
6,2024-05-04,10645,1.344651,16.5802,0.4727
7,2024-05-04,19623,1.590204,21.1333,0.6142
8,2024-05-04,25049,2.170204,10.6465,0.3024
9,2024-05-09,10645,2.371944,12.0574,0.2934


Im unteren Teil hier werden nun unabhängig von der ML-Master-Datei einfach alle Stationen Verkehrsmessstationen in 4 Spalten aufgezeigt. Der Votreil ist, dass dadurch keine Messwerte verloren gehen, denn ansonsten wird nur die IDW zu den übrigen Luftmessstationen gemessen. Es gibt nun "prox_car" & "prox_truck", die kumuliert allen "Hintergrundverkehr" in der gesamten Stadt anzeigen. Sowie "IDW_car" & "IDW_truck" die je nach diestanz zur nächsten Luftmessstation einen höheren oder niedrigeren Wert der Inverted Distance Weight angeben.

In [1]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# =========================================================
# 1. Pfade definieren
# =========================================================
base_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Sensor_Community)"
verkehr_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Verkehrsmesswerte)\Verkehrsdaten_mit_Koordinaten_merged_cars_n_trucks.csv"
sensor_raw_file = os.path.join(base_path, "sensor_community_complete_may_august.csv")

# Neuer, unabhängiger Ausgabe-Pfad für die reine Feature-Tabelle
output_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\13Uhr_Verkehrs_Proxys_Komplett.csv"

def calculate_distance_matrix_m(lat1, lon1, lat2_series, lon2_series):
    """Berechnet die Distanz in Metern von einem Punkt zu einer ganzen Serie von Koordinaten."""
    R = 6371000.0
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2_series)
    dphi = np.radians(lat2_series - lat1)
    dlambda = np.radians(lon2_series - lon1)

    a = (
        np.sin(dphi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    )
    return R * 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))

# =========================================================
# 2. Verkehrsdaten für 13:00 Uhr laden & numerisch absichern
# =========================================================
print("Lade und bereinige Verkehrsdaten (13:00 Uhr)...")
df_traffic = pd.read_csv(verkehr_file, sep=";", decimal=",")

df_traffic["Time"] = pd.to_datetime(
    df_traffic["Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce"
)
df_traffic_13 = df_traffic[df_traffic["Time"].dt.hour == 13].copy()
df_traffic_13["date"] = df_traffic_13["Time"].dt.strftime("%Y-%m-%d")

for col in ["Lat", "Lon", "processed_car_vol", "processed_truck_vol"]:
    df_traffic_13[col] = (
        df_traffic_13[col].astype(str).str.replace(",", ".", regex=False)
    )
    df_traffic_13[col] = pd.to_numeric(df_traffic_13[col], errors="coerce")

df_traffic_13 = df_traffic_13.dropna(
    subset=["Lat", "Lon", "processed_car_vol"]
)
print(f"-> {len(df_traffic_13)} Verkehrs-Beobachtungen um 13:00 Uhr geladen.")

# =========================================================
# 3. Sensor-Koordinaten extrahieren
# =========================================================
print("Extrahiere Sensor-Koordinaten...")
df_sc = pd.read_csv(sensor_raw_file, sep=";", decimal=",")

for col in ["lat", "lon"]:
    df_sc[col] = df_sc[col].astype(str).str.replace(",", ".", regex=False)
    df_sc[col] = pd.to_numeric(df_sc[col], errors="coerce")

sensor_coords = (
    df_sc[["sensor_id", "lat", "lon"]].drop_duplicates(subset=["sensor_id"]).dropna()
)
sensor_coords["station_id"] = (
    sensor_coords["sensor_id"].astype(float).astype(int).astype(str)
)
sensor_coords = sensor_coords[["station_id", "lat", "lon"]]

kopfklinik_entry = pd.DataFrame(
    [
        {
            "station_id": "1",
            "lat": 49.8047,
            "lon": 9.95642,
        }
    ]
)
all_sensors = pd.concat(
    [sensor_coords, kopfklinik_entry], ignore_index=True
).drop_duplicates(subset=["station_id"])
print(f"-> {len(all_sensors)} Sensor-Standorte verriegelt.")

# =========================================================
# 4. Proxys berechnen (Kumulativ & Max IDW)
# =========================================================
print("Berechne Proxy-Werte für alle Tage und Stationen...")
cumulative_results = []

unique_dates = df_traffic_13["date"].unique()

for current_date in unique_dates:
    daily_traffic = df_traffic_13[df_traffic_13["date"] == current_date]

    for _, sensor in all_sensors.iterrows():
        s_id = sensor["station_id"]
        distances = calculate_distance_matrix_m(
            sensor["lat"], sensor["lon"], daily_traffic["Lat"], daily_traffic["Lon"]
        )

        # IDW-Serien berechnen
        idw_car_series = daily_traffic["processed_car_vol"] / ((distances + 1.0))
        idw_truck_series = daily_traffic["processed_truck_vol"].fillna(0.0) / ((distances + 1.0))

        cumulative_results.append(
            {
                "date": current_date,
                "station_id": s_id,
                "prox_kumulativ_car": round(idw_car_series.sum(), 4),
                "prox_kumulativ_truck": round(idw_truck_series.sum(), 4),
                "IDW_car": round(idw_car_series.max(), 4),
                "IDW_truck": round(idw_truck_series.max(), 4),
            }
        )

df_proxys_final = pd.DataFrame(cumulative_results)
print(f"-> {len(df_proxys_final)} Feature-Kombinationen berechnet.")

# =========================================================
# 5. Unabhängigen Feature-Datensatz exportieren
# =========================================================
df_proxys_final.to_csv(output_file, index=False, sep=";", decimal=",")
print("-" * 60)
print(f"ERFOLG! Die eigenständige Verkehrs-Tabelle liegt hier:\n{output_file}")
print("-" * 60)
display(df_proxys_final.head(10))

Lade und bereinige Verkehrsdaten (13:00 Uhr)...
-> 11808 Verkehrs-Beobachtungen um 13:00 Uhr geladen.
Extrahiere Sensor-Koordinaten...
-> 9 Sensor-Standorte verriegelt.
Berechne Proxy-Werte für alle Tage und Stationen...
-> 1107 Feature-Kombinationen berechnet.
------------------------------------------------------------
ERFOLG! Die eigenständige Verkehrs-Tabelle liegt hier:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Exponent2_13Uhr_Verkehrs_Proxys_Komplett.csv
------------------------------------------------------------


,date,station_id,prox_kumulativ_car,prox_kumulativ_truck,IDW_car,IDW_truck
0,2024-05-01,5456,0.0216,0.0004,0.0049,0.0001
1,2024-05-01,9094,0.0040,0.0001,0.0003,0.0000
2,2024-05-01,10645,0.0110,0.0002,0.0017,0.0000
3,2024-05-01,13975,0.0241,0.0005,0.0049,0.0002
4,2024-05-01,15449,0.0228,0.0004,0.0043,0.0001
5,2024-05-01,19623,0.0168,0.0004,0.0025,0.0001
6,2024-05-01,25049,0.0030,0.0001,0.0005,0.0000
7,2024-05-01,46505,0.0085,0.0002,0.0015,0.0001
8,2024-05-01,1,0.0200,0.0004,0.0048,0.0001
9,2024-05-02,5456,0.0339,0.0027,0.0074,0.0009


Hier nochmal das selbe skript aber nicht auf 13 Uhr beschränkt.

In [2]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# =========================================================
# 1. Pfade definieren
# =========================================================
base_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Sensor_Community)"
verkehr_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Verkehrsmesswerte)\Verkehrsdaten_mit_Koordinaten_merged_cars_n_trucks.csv"
sensor_raw_file = os.path.join(base_path, "sensor_community_complete_may_august.csv")

# Ausgabe-Pfad für die eigenständige 24h-Tages-Tabelle
output_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\24Std_Verkehrs_Proxys_Komplett.csv"

def calculate_distance_matrix_m(lat1, lon1, lat2_series, lon2_series):
    """Berechnet die Distanz in Metern von einem Punkt zu einer ganzen Serie von Koordinaten."""
    R = 6371000.0
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2_series)
    dphi = np.radians(lat2_series - lat1)
    dlambda = np.radians(lon2_series - lon1)

    a = (
        np.sin(dphi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    )
    return R * 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))

# =========================================================
# 2. Verkehrsdaten für den GESAMTEN Tag (0–23 Uhr) laden & aggregieren
# =========================================================
print("Lade Verkehrsdaten für den gesamten Tag (0 bis 23 Uhr)...")
df_traffic = pd.read_csv(verkehr_file, sep=";", decimal=",")

# Zeitstempel umwandeln
df_traffic["Time"] = pd.to_datetime(
    df_traffic["Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce"
)

# Datum als YYYY-MM-DD extrahieren
df_traffic["date"] = df_traffic["Time"].dt.strftime("%Y-%m-%d")

# Zahlenspalten bereinigen und in numerische Floats konvertieren
for col in ["Lat", "Lon", "processed_car_vol", "processed_truck_vol"]:
    df_traffic[col] = (
        df_traffic[col].astype(str).str.replace(",", ".", regex=False)
    )
    df_traffic[col] = pd.to_numeric(df_traffic[col], errors="coerce")

# Zeilen ohne Koordinaten oder Datum entfernen
df_traffic = df_traffic.dropna(subset=["Lat", "Lon", "processed_car_vol", "date"])

# HIER NEU: Tages-Gesamtsumme pro Messstation & Datum berechnen (0 bis 23 Uhr)
print("Summiere Verkehrsaufkommen pro Station über alle 24 Stunden eines Tages...")
daily_traffic_aggregated = (
    df_traffic.groupby(["date", "Join_Key", "Lat", "Lon"], as_index=False)[
        ["processed_car_vol", "processed_truck_vol"]
    ]
    .sum()
)

print(f"-> {len(daily_traffic_aggregated)} Tages-Messreihen von Verkehrsmessstationen aggregiert.")

# =========================================================
# 3. Sensor-Koordinaten extrahieren
# =========================================================
print("Extrahiere Sensor-Koordinaten...")
df_sc = pd.read_csv(sensor_raw_file, sep=";", decimal=",")

for col in ["lat", "lon"]:
    df_sc[col] = df_sc[col].astype(str).str.replace(",", ".", regex=False)
    df_sc[col] = pd.to_numeric(df_sc[col], errors="coerce")

sensor_coords = (
    df_sc[["sensor_id", "lat", "lon"]].drop_duplicates(subset=["sensor_id"]).dropna()
)
sensor_coords["station_id"] = (
    sensor_coords["sensor_id"].astype(float).astype(int).astype(str)
)
sensor_coords = sensor_coords[["station_id", "lat", "lon"]]

kopfklinik_entry = pd.DataFrame(
    [
        {
            "station_id": "1",
            "lat": 49.8047,
            "lon": 9.95642,
        }
    ]
)
all_sensors = pd.concat(
    [sensor_coords, kopfklinik_entry], ignore_index=True
).drop_duplicates(subset=["station_id"])
print(f"-> {len(all_sensors)} Sensor-Standorte verriegelt.")

# =========================================================
# 4. 24h-Proxys berechnen (Kumulativ & Max IDW)
# =========================================================
print("Berechne 24h-Proxy-Werte für alle Tage und Stationen...")
cumulative_results = []

unique_dates = daily_traffic_aggregated["date"].unique()

for current_date in unique_dates:
    daily_traffic = daily_traffic_aggregated[daily_traffic_aggregated["date"] == current_date]

    for _, sensor in all_sensors.iterrows():
        s_id = sensor["station_id"]
        distances = calculate_distance_matrix_m(
            sensor["lat"], sensor["lon"], daily_traffic["Lat"], daily_traffic["Lon"]
        )

        # IDW-Serien auf Basis der 24h-Tagesgesamtsumme berechnen
        idw_car_series = daily_traffic["processed_car_vol"] / (distances + 1.0)
        idw_truck_series = daily_traffic["processed_truck_vol"].fillna(0.0) / (distances + 1.0)

        cumulative_results.append(
            {
                "date": current_date,
                "station_id": s_id,
                "prox_kumulativ_car_24h": round(idw_car_series.sum(), 4),
                "prox_kumulativ_truck_24h": round(idw_truck_series.sum(), 4),
                "IDW_car_24h": round(idw_car_series.max(), 4),
                "IDW_truck_24h": round(idw_truck_series.max(), 4),
            }
        )

df_proxys_24h = pd.DataFrame(cumulative_results)
print(f"-> {len(df_proxys_24h)} Feature-Kombinationen (24h) berechnet.")

# =========================================================
# 5. Unabhängigen 24h-Feature-Datensatz exportieren
# =========================================================
df_proxys_24h.to_csv(output_file, index=False, sep=";", decimal=",")
print("-" * 60)
print(f"ERFOLG! Die eigenständige 24h-Verkehrs-Tabelle liegt hier:\n{output_file}")
print("-" * 60)
display(df_proxys_24h.head(10))

Lade Verkehrsdaten für den gesamten Tag (0 bis 23 Uhr)...
Summiere Verkehrsaufkommen pro Station über alle 24 Stunden eines Tages...
-> 11656 Tages-Messreihen von Verkehrsmessstationen aggregiert.
Extrahiere Sensor-Koordinaten...
-> 9 Sensor-Standorte verriegelt.
Berechne 24h-Proxy-Werte für alle Tage und Stationen...
-> 1116 Feature-Kombinationen (24h) berechnet.
------------------------------------------------------------
ERFOLG! Die eigenständige 24h-Verkehrs-Tabelle liegt hier:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\24Std_Verkehrs_Proxys_Komplett.csv
------------------------------------------------------------


,date,station_id,prox_kumulativ_car_24h,prox_kumulativ_truck_24h,IDW_car_24h,IDW_truck_24h
0,2024-05-01,5456,214.7011,4.4747,23.2269,0.4496
1,2024-05-01,9094,106.1205,2.0537,6.3053,0.1698
2,2024-05-01,10645,155.4266,3.1769,13.5919,0.4892
3,2024-05-01,13975,216.2144,4.6890,21.8070,1.1455
4,2024-05-01,15449,189.9150,3.8663,19.1147,0.6331
5,2024-05-01,19623,198.3082,4.2554,14.5666,0.6020
6,2024-05-01,25049,96.7378,1.9476,7.5525,0.1599
7,2024-05-01,46505,137.0316,2.9972,16.4576,0.7015
8,2024-05-01,1,192.7245,4.0554,40.3317,0.8420
9,2024-05-02,5456,384.2743,20.1842,42.8412,3.2108
